# iSign Track 1 · Notebook 1 — download the paper's "Part AD" subset

Reproduces the data of **arXiv 2609.12993** ("Investigating Temporal Motion Features for
Pose-to-Text ISL Translation"): *"we use the iSign poses v1.1 Part AD subset, containing pose
representations for 18,867 sign language samples."*

iSign stores all poses as one 170 GB zip split into 4 parts. This notebook reads the zip's
index from the end of the last part and downloads **only the part-AD files** with HTTP range
requests, so nothing large is ever stored. Each `.pose` file is converted with the paper's own
`load_pose_file` into a `(frames, 300)` float32 array (33 body + 21 left-hand + 21 right-hand
landmarks × x, y, z, confidence) and saved as `<uid>.npy`.

### Before running
1. On https://huggingface.co/datasets/Exploration-Lab/iSign click **"Agree and access"**
   (auto-approved; licence CC-BY-NC-SA-4.0, non-commercial).
2. Create a **read** token at https://huggingface.co/settings/tokens
3. In this notebook: **Add-ons → Secrets → Add secret**, label `HF_TOKEN`, paste the token,
   and tick it for this notebook.
4. Settings (right panel): **Accelerator: None** (CPU only, saves GPU quota), **Internet: On**.
5. **Save Version → Save & Run All (Commit)**. About 30–90 min (it downloads about 25 GB).

### After it finishes
Open the finished version → **Output** → **New Dataset** → name it **`isign-partad-features`**
(private). Notebook 2 trains on that dataset.

In [ ]:
!pip install -q "pose-format==0.15.0"

### Source files (identical to the locally tested versions)

In [ ]:
%%writefile paper_pose_utils.py
# VERBATIM copy of inference/pose_utils.py from huggingface.co/manavdhamecha77/iSign-t5-pose-to-text
# (paper arXiv 2609.12993). Do not edit: this is the exact preprocessing the paper used.
"""Utilities for reading and preprocessing iSign binary .pose files."""

import logging
import json
from typing import List, Optional, Sequence, Tuple

import numpy as np

logger = logging.getLogger(__name__)

DEFAULT_COMPONENTS = (
    "POSE_LANDMARKS",
    "LEFT_HAND_LANDMARKS",
    "RIGHT_HAND_LANDMARKS",
)


def load_pose_file(
    pose_path: str,
    components: Optional[Sequence[str]] = DEFAULT_COMPONENTS,
    include_confidence: bool = True,
) -> np.ndarray:

    try:
        with open(pose_path, "rb") as f:
            raw = f.read()

        if raw.lstrip().startswith((b"{", b"[")):
            payload = json.loads(raw.decode("utf-8"))
            frames = (
                payload.get("data", [])
                if isinstance(payload, dict)
                else payload
            )

            rows = [
                np.asarray(
                    frame.get("keypoints", []),
                    dtype=np.float32,
                ).reshape(-1)
                for frame in frames
            ]

            return (
                np.asarray(rows, dtype=np.float32)
                if rows
                else np.array([])
            )

    except (
        json.JSONDecodeError,
        UnicodeDecodeError,
        FileNotFoundError,
        OSError,
    ) as exc:
        logger.error(
            "Failed to read pose file %s: %s",
            pose_path,
            exc,
        )
        return np.array([])

    try:
        from pose_format import Pose

        pose = Pose.read(raw)

    except (
        FileNotFoundError,
        ImportError,
        ValueError,
        OSError,
    ) as exc:
        logger.error(
            "Failed to load pose file %s: %s",
            pose_path,
            exc,
        )
        return np.array([])

    data = np.asarray(
        pose.body.data,
        dtype=np.float32,
    )

    confidence = np.asarray(
        pose.body.confidence,
        dtype=np.float32,
    )

    if data.ndim != 4 or data.shape[1] == 0:
        logger.warning(
            "Unexpected pose shape in %s: %s",
            pose_path,
            data.shape,
        )
        return np.array([])

    data = data[:, 0]

    confidence = (
        confidence[:, 0]
        if confidence.ndim == 3
        else None
    )

    wanted = (
        set(components)
        if components is not None
        else None
    )

    selected = []

    offset = 0

    for component in pose.header.components:

        count = len(component.points)

        if (
            wanted is None
            or component.name in wanted
        ):

            component_data = data[
                :,
                offset:offset + count,
                :,
            ]

            if (
                include_confidence
                and confidence is not None
            ):

                component_data = np.concatenate(
                    [
                        component_data,
                        confidence[
                            :,
                            offset:offset + count,
                            None,
                        ],
                    ],
                    axis=2,
                )

            selected.append(component_data)

        offset += count

    if not selected:

        logger.warning(
            "No requested components found in %s",
            pose_path,
        )

        return np.array([])

    return np.concatenate(
        selected,
        axis=1,
    ).reshape(
        data.shape[0],
        -1,
    )


def normalize_pose(
    pose: np.ndarray,
    method: str = "zscore",
) -> np.ndarray:

    if pose.size == 0:
        return pose

    if method == "zscore":

        mean = np.nanmean(
            pose,
            axis=0,
            keepdims=True,
        )

        std = np.nanstd(
            pose,
            axis=0,
            keepdims=True,
        )

        return np.nan_to_num(
            (pose - mean)
            / np.where(
                std == 0,
                1.0,
                std,
            )
        )

    if method == "minmax":

        min_val = np.nanmin(
            pose,
            axis=0,
            keepdims=True,
        )

        max_val = np.nanmax(
            pose,
            axis=0,
            keepdims=True,
        )

        return np.nan_to_num(
            (pose - min_val)
            / np.where(
                max_val == min_val,
                1.0,
                max_val - min_val,
            )
        )

    raise ValueError(
        f"Unknown normalization method: {method}"
    )


def pad_or_truncate_pose(
    pose: np.ndarray,
    max_length: int,
    pad_value: float = 0.0,
) -> Tuple[np.ndarray, int]:

    if pose.size == 0:

        width = (
            pose.shape[1]
            if pose.ndim > 1
            else 0
        )

        return (
            np.full(
                (max_length, width),
                pad_value,
                dtype=np.float32,
            ),
            0,
        )

    length = min(
        pose.shape[0],
        max_length,
    )

    padded = np.full(
        (max_length, pose.shape[1]),
        pad_value,
        dtype=pose.dtype,
    )

    padded[:length] = pose[:length]

    return padded, length


def add_motion_features(
    pose: np.ndarray,
) -> np.ndarray:
    """
    Concatenate frame-wise velocity with pose features.

    velocity[t] = pose[t] - pose[t - 1]

    First frame velocity is zero.
    """

    if pose.size == 0:
        return pose

    velocity = np.zeros_like(
        pose,
        dtype=np.float32,
    )

    velocity[1:] = (
        pose[1:] - pose[:-1]
    )

    return np.concatenate(
        [
            pose,
            velocity,
        ],
        axis=1,
    )


def preprocess_pose(
    pose_path: str,
    max_length: int,
    normalization: str = "zscore",
    components: Optional[
        Sequence[str]
    ] = DEFAULT_COMPONENTS,
    include_confidence: bool = True,
    pose_dim: int = 600,
    use_motion: bool = True,
) -> Tuple[np.ndarray, int]:

    pose = load_pose_file(
        pose_path,
        components,
        include_confidence,
    )

    base_pose_dim = (
        pose_dim // 2
        if use_motion
        else pose_dim
    )

    if pose.size == 0:

        return (
            np.zeros(
                (
                    max_length,
                    pose_dim,
                ),
                dtype=np.float32,
            ),
            0,
        )

    if pose.shape[1] < base_pose_dim:

        pose = np.pad(
            pose,
            (
                (0, 0),
                (
                    0,
                    base_pose_dim
                    - pose.shape[1],
                ),
            ),
        )

    elif pose.shape[1] > base_pose_dim:

        pose = pose[
            :,
            :base_pose_dim,
        ]

    pose = normalize_pose(
        pose,
        normalization,
    )

    if use_motion:

        pose = add_motion_features(
            pose,
        )

    return pad_or_truncate_pose(
        pose,
        max_length,
    )


def create_attention_mask(
    actual_lengths: List[int],
    max_length: int,
) -> np.ndarray:

    mask = np.zeros(
        (
            len(actual_lengths),
            max_length,
        ),
        dtype=np.int64,
    )

    for index, length in enumerate(
        actual_lengths
    ):

        mask[
            index,
            :length,
        ] = 1

    return mask

In [ ]:
%%writefile isign_zip.py
"""Random access into the iSign pose archive without downloading all 170 GB.

iSign ships its poses as ONE zip file split into four parts
(iSign-poses_v1.1_part_aa .. _ad; the dataset card says to `cat` them together).
A zip keeps its index (the "central directory") at the very end, so we can:

  1. treat the four parts as one virtual file, read via HTTP Range requests;
  2. let Python's zipfile parse the central directory from the tail of part_ad;
  3. fetch any member file on its own with one Range request.

The paper (arXiv 2609.12993) trained on "the iSign poses v1.1 Part AD subset"
(18,867 samples): the members whose data lives in part_ad. `members_in_part`
reproduces exactly that selection.
"""

import io
import random
import struct
import threading
import time
import zipfile
import zlib

import requests

HF_REPO = "Exploration-Lab/iSign"
PART_NAMES = [f"iSign-poses_v1.1_part_{s}" for s in ("aa", "ab", "ac", "ad")]


def hf_url(filename, repo=HF_REPO):
    return f"https://huggingface.co/datasets/{repo}/resolve/main/{filename}"


def part_sizes(token, repo=HF_REPO):
    """Exact byte size of each part, from the Hub API."""
    r = requests.get(f"https://huggingface.co/api/datasets/{repo}/tree/main",
                     headers={"Authorization": f"Bearer {token}"}, timeout=60)
    r.raise_for_status()
    sizes = {f["path"]: f["size"] for f in r.json()}
    return [sizes[name] for name in PART_NAMES]


class MultiPartSource:
    """Byte-range reader over several consecutive parts (HTTP or local files)."""

    def __init__(self, parts, sizes, fetch):
        # parts: list of identifiers; sizes: their byte sizes;
        # fetch(part, start, end_inclusive) -> bytes
        self.parts, self.sizes, self.fetch = parts, sizes, fetch
        self.starts = []
        pos = 0
        for s in sizes:
            self.starts.append(pos)
            pos += s
        self.total = pos

    def part_start(self, index):
        return self.starts[index]

    def read_range(self, start, length):
        """Read `length` bytes at virtual offset `start`, crossing parts if needed."""
        out, pos, end = [], start, min(start + length, self.total)
        while pos < end:
            i = max(k for k, s in enumerate(self.starts) if s <= pos)
            local = pos - self.starts[i]
            take = min(end - pos, self.sizes[i] - local)
            out.append(self.fetch(self.parts[i], local, local + take - 1))
            pos += take
        return b"".join(out)


def http_fetcher(token, retries=10):
    """fetch(part, start, end) via HTTP Range on the gated HF dataset.

    HuggingFace answers HTTP 429 when too many requests arrive at once; we then
    wait as long as its Retry-After header asks (or back off exponentially).
    One requests.Session per thread, since sessions aren't guaranteed thread-safe.
    """
    local = threading.local()

    def session():
        if not hasattr(local, "s"):
            local.s = requests.Session()
            local.s.headers["Authorization"] = f"Bearer {token}"
        return local.s

    def fetch(part, start, end):
        want = end - start + 1
        for attempt in range(retries):
            try:
                r = session().get(hf_url(part), headers={"Range": f"bytes={start}-{end}"},
                                  timeout=120)
                if r.status_code == 206 and len(r.content) == want:
                    return r.content
                if r.status_code in (401, 403):
                    raise PermissionError(
                        f"HTTP {r.status_code}: accept the iSign terms on HuggingFace and "
                        "check your HF token.")
                if r.status_code == 429:
                    wait = float(r.headers.get("Retry-After", 0) or 0) or min(60, 5 * 2 ** attempt)
                    time.sleep(wait + random.random() * 3)
                    continue
                raise IOError(f"HTTP {r.status_code}, got {len(r.content)}/{want} bytes")
            except PermissionError:
                raise
            except Exception:
                if attempt == retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        raise IOError(f"still rate-limited after {retries} tries ({part} {start}-{end})")
    return fetch


def local_fetcher():
    """fetch(path, start, end) from local files (used by the offline tests)."""
    def fetch(path, start, end):
        with open(path, "rb") as f:
            f.seek(start)
            return f.read(end - start + 1)
    return fetch


class _SeekableView(io.RawIOBase):
    """Minimal seekable file over a MultiPartSource, just so zipfile can parse
    the central directory. Reads are cached in 8 MB blocks."""

    BLOCK = 8 << 20

    def __init__(self, source):
        self.src, self.pos, self.cache = source, 0, {}

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        base = {0: 0, 1: self.pos, 2: self.src.total}[whence]
        self.pos = base + offset
        return self.pos

    def _block(self, b):
        if b not in self.cache:
            self.cache[b] = self.src.read_range(b * self.BLOCK, self.BLOCK)
        return self.cache[b]

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.src.total - self.pos
        out = bytearray()
        while n > 0 and self.pos < self.src.total:
            b, off = divmod(self.pos, self.BLOCK)
            chunk = self._block(b)[off:off + n]
            if not chunk:
                break
            out += chunk
            self.pos += len(chunk)
            n -= len(chunk)
        return bytes(out)

    def readinto(self, buf):
        data = self.read(len(buf))
        buf[:len(data)] = data
        return len(data)


def list_members(source):
    """All zip members (zipfile.ZipInfo), parsed from the archive's tail."""
    with zipfile.ZipFile(_SeekableView(source)) as zf:
        return [i for i in zf.infolist() if not i.is_dir()]


def members_in_part(members, source, part_index):
    """Members whose local header starts inside part `part_index` (the paper's
    "Part AD subset" is part_index=3)."""
    lo = source.part_start(part_index)
    hi = lo + source.sizes[part_index]
    return [m for m in members if lo <= m.header_offset < hi]


def read_member(source, info):
    """Bytes of one member, with a single range request (plus one if the local
    header's extra field is unusually large)."""
    guess = 30 + len(info.filename.encode("utf-8")) + 256
    head = source.read_range(info.header_offset, guess + info.compress_size)
    if head[:4] != b"PK\x03\x04":
        raise IOError(f"bad local header for {info.filename}")
    name_len, extra_len = struct.unpack("<HH", head[26:30])
    data_start = 30 + name_len + extra_len
    need = data_start + info.compress_size
    if need > len(head):
        head = source.read_range(info.header_offset, need)
    raw = head[data_start:need]
    if info.compress_type == zipfile.ZIP_STORED:
        data = raw
    elif info.compress_type == zipfile.ZIP_DEFLATED:
        data = zlib.decompress(raw, -15)
    else:
        raise NotImplementedError(f"compression type {info.compress_type}")
    if zlib.crc32(data) & 0xFFFFFFFF != info.CRC:
        raise IOError(f"CRC mismatch for {info.filename}")
    return data

In [ ]:
%%writefile features.py
"""Turn iSign .pose bytes into the paper's raw per-frame feature matrix.

We call the paper's own `load_pose_file` so the numbers are identical to what
their training code saw: POSE(33) + LEFT_HAND(21) + RIGHT_HAND(21) landmarks,
each (x, y, z, confidence) -> (frames, 300) float32.

Everything after loading (z-score, motion, padding to 500 frames) is applied at
training time by `preprocess_array`, so one cached feature file serves both the
spatial (300-d) and motion (600-d) models.
"""

import os
import tempfile

import numpy as np

from paper_pose_utils import (
    DEFAULT_COMPONENTS,
    add_motion_features,
    load_pose_file,
    normalize_pose,
    pad_or_truncate_pose,
)


def pose_bytes_to_features(pose_bytes):
    """Exactly `load_pose_file(path, DEFAULT_COMPONENTS, include_confidence=True)`."""
    fd, path = tempfile.mkstemp(suffix=".pose")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(pose_bytes)
        return load_pose_file(path, DEFAULT_COMPONENTS, True)
    finally:
        os.remove(path)


def preprocess_array(pose, max_length=500, normalization="zscore", pose_dim=300,
                     use_motion=False):
    """Same steps, in the same order, as the paper's `preprocess_pose` after the
    file has been loaded (see paper_pose_utils.preprocess_pose)."""
    base_pose_dim = pose_dim // 2 if use_motion else pose_dim
    if pose.size == 0:
        return np.zeros((max_length, pose_dim), dtype=np.float32), 0
    if pose.shape[1] < base_pose_dim:
        pose = np.pad(pose, ((0, 0), (0, base_pose_dim - pose.shape[1])))
    elif pose.shape[1] > base_pose_dim:
        pose = pose[:, :base_pose_dim]
    pose = normalize_pose(pose, normalization)
    if use_motion:
        pose = add_motion_features(pose)
    return pad_or_truncate_pose(pose, max_length)

### Configuration

In [ ]:
import os, sys, time, json
import numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
sys.path.insert(0, ".")
import isign_zip
from features import pose_bytes_to_features

OUT = "/kaggle/working/isign_partad_features"
os.makedirs(OUT, exist_ok=True)
PART_INDEX = 3      # 0=aa, 1=ab, 2=ac, 3=ad. The paper used part AD.
THREADS = 6         # parallel range downloads (16 triggered HTTP 429 rate limits)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

### Step 1 · annotations CSV

In [ ]:
r = requests.get(isign_zip.hf_url("iSign_v1.1.csv"),
                 headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=120)
if r.status_code in (401, 403):
    raise PermissionError("Accept the iSign terms on HuggingFace and check the HF_TOKEN secret.")
r.raise_for_status()
open(f"{OUT}/iSign_v1.1.csv", "wb").write(r.content)
csv_df = pd.read_csv(f"{OUT}/iSign_v1.1.csv")
print(csv_df.shape, list(csv_df.columns))
csv_df.head()

### Step 2 · read the zip index and select the part-AD members

In [ ]:
sizes = isign_zip.part_sizes(HF_TOKEN)
print("part sizes (GB):", [round(s / 1e9, 2) for s in sizes])
src = isign_zip.MultiPartSource(isign_zip.PART_NAMES, sizes, isign_zip.http_fetcher(HF_TOKEN))

t = time.time()
members = isign_zip.list_members(src)
print(f"{len(members):,} files in the whole archive ({time.time()-t:.0f}s)")
pose_members = [m for m in members if m.filename.endswith(".pose")]
selected = isign_zip.members_in_part(pose_members, src, PART_INDEX)
print(f"{len(selected):,} .pose files in part {isign_zip.PART_NAMES[PART_INDEX]}  "
      "(paper reports 18,867)")
print("example names:", [m.filename for m in selected[:3]])
print("GB to download:", round(sum(m.compress_size for m in selected) / 1e9, 2))

uid_of = lambda m: os.path.splitext(os.path.basename(m.filename))[0]
in_csv = set(csv_df["uid"].astype(str))
print("selected files with a CSV row:", sum(uid_of(m) in in_csv for m in selected))

### Step 3 · download + convert (resumable: already-saved files are skipped)

In [ ]:
def work(m):
    uid = uid_of(m)
    path = f"{OUT}/{uid}.npy"
    if os.path.exists(path):
        return uid, "exists", None
    try:
        feats = pose_bytes_to_features(isign_zip.read_member(src, m))
        if feats.size == 0:
            return uid, "empty", 0
        np.save(path, feats.astype(np.float32))
        return uid, "ok", feats.shape[0]
    except Exception as e:
        return uid, f"error: {type(e).__name__}: {e}", None

rows, t = [], time.time()
with ThreadPoolExecutor(THREADS) as pool:
    futures = [pool.submit(work, m) for m in selected]
    for i, f in enumerate(as_completed(futures), 1):
        rows.append(f.result())
        if i % 500 == 0 or i == len(futures):
            ok = sum(r[1] in ("ok", "exists") for r in rows)
            print(f"{i:,}/{len(futures):,} done, {ok:,} saved, {(time.time()-t)/60:.1f} min",
                  flush=True)

status = pd.DataFrame(rows, columns=["uid", "status", "frames"])
status.to_csv(f"{OUT}/download_status.csv", index=False)
print(status.status.str.split(":").str[0].value_counts())

### Step 4 · summary

In [ ]:
saved = status[status.status.isin(["ok", "exists"])]
print(f"saved feature files: {len(saved):,}")
print("frames per sample:", saved.frames.dropna().describe().round(1).to_dict())
print("output size (GB):", round(sum(os.path.getsize(os.path.join(OUT, f))
                                     for f in os.listdir(OUT)) / 1e9, 2))
print(status[~status.status.isin(["ok", "exists"])].head(20))